# PACE Level 2
<br>

**Steps:**

* Create a plan for files to use `pc.plan()`
* Print the plan to check it `print(plan.summary())`
* Do the plan and get matchups `pc.matchup(plan, spatial_method="xoak")`

## Prerequisite -- Login to EarthData

The examples here use NASA EarthData and you need to have an account with EarthData. Make sure you can login.

In [4]:
import earthaccess
import xoak
earthaccess.login()

## Here are the level 2 datasets

In [5]:
import earthaccess
results = earthaccess.search_datasets(instrument="oci")

short_names = [
    item.summary()["short-name"]
    for item in results
    if "L2" in item.summary()["short-name"]
]

print(short_names)

['PACE_OCI_L2_UVAI_UAA_NRT', 'PACE_OCI_L2_UVAI_UAA', 'PACE_OCI_L2_AER_UAA_NRT', 'PACE_OCI_L2_AER_UAA', 'PACE_OCI_L2_AOP_NRT', 'PACE_OCI_L2_AOP', 'PACE_OCI_L2_CLOUD_MASK_NRT', 'PACE_OCI_L2_CLOUD_MASK', 'PACE_OCI_L2_CLOUD_NRT', 'PACE_OCI_L2_CLOUD', 'PACE_OCI_L2_LANDVI_NRT', 'PACE_OCI_L2_LANDVI', 'PACE_OCI_L2_BGC_NRT', 'PACE_OCI_L2_BGC', 'PACE_OCI_L2_IOP_NRT', 'PACE_OCI_L2_IOP', 'PACE_OCI_L2_PAR_NRT', 'PACE_OCI_L2_PAR', 'PACE_OCI_L2_SFREFL_NRT', 'PACE_OCI_L2_SFREFL', 'PACE_OCI_L2_TRGAS_NRT', 'PACE_OCI_L2_TRGAS']


## Load some points

In [1]:
import pandas as pd
url = (
    "https://raw.githubusercontent.com/"
    "fish-pace/point-collocation/main/"
    "examples/fixtures/points.csv"
)
df_points = pd.read_csv(url)
print(len(df_points))
df_points.head()

595


,lat,lon,date
0,27.3835,-82.7375,2024-06-13
1,27.1190,-82.7125,2024-06-14
2,26.9435,-82.8170,2024-06-14
3,26.6875,-82.8065,2024-06-14
4,26.6675,-82.6455,2024-06-14


## Get a plan for matchups for 1st 50 points from PACE data

In [2]:
%%time
import point_collocation as pc
plan = pc.plan(
    df_points[0:50], 	
    data_source="earthaccess",
    source_kwargs={
        "short_name": "PACE_OCI_L2_AOP",
    },
    time_buffer="12h"
)

CPU times: user 640 ms, sys: 61 ms, total: 701 ms
Wall time: 9.02 s


In [3]:
plan.summary()

Plan: 50 points → 13 unique granule(s)
  Points with 0 matches : 0
  Points with >1 matches: 10
  Time buffer: 0 days 12:00:00

First 5 point(s):
  [0] lat=27.3835, lon=-82.7375, time=2024-06-13 12:00:00: 2 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240613T171620.L2.OC_AOP.V3_1.nc
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240613T184939.L2.OC_AOP.V3_1.nc
  [1] lat=27.1190, lon=-82.7125, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240614T175104.L2.OC_AOP.V3_1.nc
  [2] lat=26.9435, lon=-82.8170, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240614T175104.L2.OC_AOP.V3_1.nc
  [3] lat=26.6875, lon=-82.8065, time=2024-06-14 12:00:00: 1 match(es)
    → https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240614T175104.L2.OC_AOP.V3_

In [11]:
%%time
# This uses open_method="auto". It will try xr.open_dataset
# discover no lat/lon and then try xr.open_datatree + merge. 
# If you know, the netcdfs are grouped, you can pass in
# open_method="datatree-merge" yourself
plan.show_variables()

open_method: {'xarray_open': 'datatree', 'open_kwargs': {'chunks': {}, 'engine': 'h5netcdf', 'decode_timedelta': False}, 'coords': 'auto', 'set_coords': True, 'dim_renames': None, 'auto_align_phony_dims': None, 'merge': 'all', 'merge_kwargs': {}}

Dimensions: {'number_of_bands': 286, 'number_of_reflective_bands': 286, 'wavelength_3d': 172, 'number_of_lines': 1710, 'pixels_per_line': 1272}

Variables: ['wavelength', 'vcal_gain', 'vcal_offset', 'F0', 'aw', 'bbw', 'k_oz', 'k_no2', 'Tau_r', 'year', 'day', 'msec', 'time', 'detnum', 'mside', 'slon', 'clon', 'elon', 'slat', 'clat', 'elat', 'csol_z', 'Rrs', 'Rrs_unc', 'aot_865', 'angstrom', 'avw', 'nflh', 'l2_flags', 'longitude', 'latitude', 'tilt']

Geolocation: ('longitude', 'latitude') — lon dims=('number_of_lines', 'pixels_per_line'), lat dims=('number_of_lines', 'pixels_per_line')

DataTree groups (detail):
  /
    Dimensions: {}
    Variables: []
  /sensor_band_parameters
    Dimensions: {'number_of_bands': 286, 'number_of_reflective_ban

## Get the matchups using that plan

`pc.matchup()` with `open_method="datatree-merge"` opens each L2 granule as a DataTree and merges all groups into a flat dataset. Use `spatial_method="xoak"` for 2-D swath geolocation. I turn on `batch_size=5` and `silent=False` to watch the progress.

Notice, that point 0 is matched to 2 granules and so has 2 rows with the same `pc_id`.

In [6]:
%%time
res = pc.matchup(plan, spatial_method="xoak", variables=["Rrs"])

CPU times: user 32.3 s, sys: 2.28 s, total: 34.6 s
Wall time: 58.9 s


In [8]:
res.head()

,lat,lon,time,pc_id,granule_id,granule_time,granule_lat,granule_lon,Rrs_346,Rrs_348,...,Rrs_706,Rrs_707,Rrs_708,Rrs_709,Rrs_711,Rrs_712,Rrs_713,Rrs_714,Rrs_717,Rrs_719
0,27.3835,-82.7375,2024-06-13 12:00:00,0,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,2024-06-13 17:18:49+00:00,27.443144,-82.612923,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,27.3835,-82.7375,2024-06-13 12:00:00,0,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,2024-06-13 18:52:08+00:00,27.383293,-82.721527,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,27.1190,-82.7125,2024-06-14 12:00:00,1,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,2024-06-14 17:53:34+00:00,27.101389,-82.717186,0.01299,0.012946,...,0.000238,0.000228,0.000198,0.000194,0.000186,0.000172,0.000152,0.000122,0.000108,0.000094
3,26.9435,-82.8170,2024-06-14 12:00:00,2,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,2024-06-14 17:53:34+00:00,26.954554,-82.810219,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,26.6875,-82.8065,2024-06-14 12:00:00,3,https://obdaac-tea.earthdatacloud.nasa.gov/ob-...,2024-06-14 17:53:34+00:00,26.703817,-82.817726,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Predefined profiles for opening granules

Granules that have groups can be opened with `xr.open_datatree()` but the user will need to specify how the groups are merged so that the lat, lon and variables can be found.  `point-collocation` has predefined profiles that you can use or modify.

In [10]:
import point_collocation.profiles as pf
pf.pace_l2

{'xarray_open': 'datatree', 'merge': 'all'}

You could modify this for PACE level 2 netcdfs by telling it to only merge the relevant groups. This doesn't actually affect speed or performance in this case.

In [13]:
test = pf.pace_l2
test['merge'] = ['/geophysical_data', '/navigation_data']

Pass to `open_method`:

In [14]:
%%time
out = pc.matchup(plan, open_method=test, variables=["Rrs"],
                     spatial_method="xoak")

CPU times: user 28 s, sys: 1.41 s, total: 29.4 s
Wall time: 52 s


In [15]:
plan.show_variables(open_method=test)

open_method: {'xarray_open': 'datatree', 'merge': ['/geophysical_data', '/navigation_data'], 'open_kwargs': {'chunks': {}, 'engine': 'h5netcdf', 'decode_timedelta': False}, 'coords': 'auto', 'set_coords': True, 'dim_renames': None, 'auto_align_phony_dims': None, 'merge_kwargs': {}}

Dimensions: {'number_of_lines': 1710, 'pixels_per_line': 1272, 'wavelength_3d': 172}

Variables: ['Rrs', 'Rrs_unc', 'aot_865', 'angstrom', 'avw', 'nflh', 'l2_flags', 'longitude', 'latitude', 'tilt']

Geolocation: ('longitude', 'latitude') — lon dims=('number_of_lines', 'pixels_per_line'), lat dims=('number_of_lines', 'pixels_per_line')

DataTree groups (detail):
  /
    Dimensions: {}
    Variables: []
  /sensor_band_parameters
    Dimensions: {'number_of_bands': 286, 'number_of_reflective_bands': 286, 'wavelength_3d': 172}
    Variables: ['wavelength', 'vcal_gain', 'vcal_offset', 'F0', 'aw', 'bbw', 'k_oz', 'k_no2', 'Tau_r']
  /scan_line_attributes
    Dimensions: {'number_of_lines': 1710}
    Variables: ['